После попыток обучить модели по фингерпринтам предполагаю, что их мало для хоть сколько-то точного предсказания токсичности (метрика точности ни разу не показала что-то выше 0.1). Я попытался что-то сделать с дисбалансом классов, но мне не хватило понимания, что нужно делать, и времени для того, чтобы разобраться.

In [9]:
# Блок импортов
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from scipy.stats import randint
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve



import warnings
warnings.filterwarnings('ignore')

In [2]:
# Загрузка данных

# Мульти-таргет, состоящий из smiles и бинарному показателю токсичности по различным категориям 
multi_target = pd.read_csv('multi_target_cleaned.csv')

# Morgan Fingerprints, состоящий из smiles и 2048 бинарных признаков для иденттификации по наличию различных химических групп в веществе
data = np.load('morgan_fp.npz')
fp_smiles = pd.read_csv('fp_smiles.csv')
sparse_matrix = csr_matrix(
    (data['data'], data['indices'], data['indptr']),
    shape=tuple(data['shape'])
)

matrix_df = pd.DataFrame.sparse.from_spmatrix(sparse_matrix)
morgan_fingerprints = pd.concat([fp_smiles, matrix_df], axis=1)

В нашем случае стоит задача предсказания токсичности химических соединений.
По имеющимся данным можно определять не только токсичность, но и её вид.
Для начала обойдёмся только токсичностью, для этого приведём мульти-таргетный датасет к одиночному бинарному показателю токсичности

In [3]:
# Берём smiles из мульти-таргетного датасета
multi_target_smiles = multi_target[['smiles']]

# Получаем список всех видов токсичности из мульти-таргетного датасета
column_list = multi_target.columns.tolist()
column_list.remove('smiles')

# Создаём датасет-шаблон для заполнения
df_tox = pd.DataFrame([{'smiles': 'sample', 'toxic': 'sample'}])
df_sample = pd.merge(multi_target_smiles, df_tox, on='smiles', how='outer')
df_unified = df_sample[df_sample['smiles'] != 'sample']

# Заполняем пустой датасет-шаблон smiles'ами и бинарным показателем токсичности, выведенным по принципу "токтично в чём-то, значит, токсично в целом"
singular_toxicity = pd.DataFrame()
for column in column_list:
    df_temp = multi_target[['smiles', column]].dropna(subset=[column])
    df_temp.rename(columns={column: 'toxic'}, inplace=True)
    merged = pd.merge(df_unified, df_temp, on='smiles', how='outer', suffixes=('_1', '_2'))
    merged['toxic_1'] = merged['toxic_1'].fillna(0)
    merged['toxic_2'] = merged['toxic_2'].fillna(0)
    merged['toxic'] = merged[['toxic_1', 'toxic_2']].max(axis=1)
    singular_toxicity = merged[['smiles', 'toxic']]
singular_toxicity = singular_toxicity.dropna()

# Создаём наш финальный датасет из fingerprint'ов и показателя токсичности
dataset = pd.merge(singular_toxicity, morgan_fingerprints, on='smiles')
dataset = dataset.drop('index', axis=1)

# Разбиваем этот датасет на матрицу fingerprint'ов (X) и вектор токсичности (y)
X = dataset.drop(['smiles', 'toxic'], axis=1).astype(int)
y = dataset[['toxic']]

In [4]:
# Здесь можем видеть, что из 339к молекул меньше 0.2% являются токисчными
y.describe()

,toxic
count,339055.000000
mean,0.001896
std,0.043507
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000


С этого момента я перестал комментировать код и хоть сколько-то следить за форматированием, потому что не хватает времени.

При имеющихся данных перед нами стоит задача классификации.
Тренируем модель:

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

rfc_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rfc_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


Проверяем метрики:

In [37]:
y_pred = rfc_model.predict(X_test)

precision_metric = precision_score(y_test, y_pred, zero_division=0)
recall_metric = recall_score(y_test, y_pred, zero_division=0)
f1_metric = f1_score(y_test, y_pred, zero_division=0)

print(f"precision: {precision_metric:.3f}; recall: {recall_metric:.3f}; f1: {f1_metric:.3f}")

precision: 0.023; recall: 0.012; f1: 0.016


Проверяем предсказания модели:

In [25]:
print(np.unique(y_pred))

[0. 1.]


Пробуем перенастроить веса:

In [22]:
n_samples = len(y_train)
n_positives = y_train.sum()
n_negatives = n_samples - n_positives
weight_positive = n_samples / (2 * n_positives)
weight_negative = n_samples / (2 * n_negatives)
class_weights = {0: weight_negative, 1: weight_positive}

rfc_model_2 = RandomForestClassifier(
    n_estimators=200,
    class_weight=class_weights,
    random_state=42,
    n_jobs=-1
)
rfc_model_2.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [23]:
y_pred_2 = rfc_model_2.predict(X_test)

precision_metric_2 = precision_score(y_test, y_pred_2, zero_division=0)
recall_metric_2 = recall_score(y_test, y_pred_2, zero_division=0)
f1_metric_2 = f1_score(y_test, y_pred_2, zero_division=0)

print(f"precision: {precision_metric_2:.3f}; recall: {recall_metric_2:.3f}; f1: {f1_metric_2:.3f}")

precision: 0.022; recall: 0.012; f1: 0.016


In [26]:
print(np.unique(y_pred_2))

[0. 1.]


In [36]:
y_pred_proba = rfc_model.predict_proba(X_test)
precision_vals, recall_vals, thresholds = precision_recall_curve(y_test, y_pred)

viable_indices = np.where(precision_vals > 0.1)[0]
if len(viable_indices) > 0:
    optimal_idx = viable_indices[np.argmax(recall_vals[viable_indices])]
    optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.01
    
    y_pred = (y_pred_proba >= optimal_threshold).astype(int)
    print(f"Оптимальный порог: {optimal_threshold:.3f}")

Оптимальный порог: 0.010


In [38]:
from sklearn.base import BaseEstimator, ClassifierMixin

class CustomThresholdClassifier(BaseEstimator, ClassifierMixin):
        
    def __init__(self, base_model, threshold=0.5):
        self.base_model = base_model
        self.threshold = threshold
    
    def fit(self, X, y, **kwargs):
        self.base_model.fit(X, y, **kwargs)
        return self
    
    def predict_proba(self, X):
        return self.base_model.predict_proba(X)
    
    def predict(self, X):
        y_proba = self.predict_proba(X)[:, 1]
        return (y_proba >= self.threshold).astype(int)
    
    def set_threshold(self, new_threshold):
        self.threshold = new_threshold
        return self

model_with_threshold = CustomThresholdClassifier(
    base_model=rfc_model,
    threshold=0.01
)

model_with_threshold.fit(X_train, y_train)

y_pred_threshold = model_with_threshold.predict(X_test)

In [39]:
model_with_threshold.set_threshold(0.015)
y_pred = model_with_threshold.predict(X_test)

In [40]:
precision_metric_threshold = precision_score(y_test, y_pred_threshold, zero_division=0)
recall_metric_threshold = recall_score(y_test, y_pred_threshold, zero_division=0)
f1_metric_threshold = f1_score(y_test, y_pred_threshold, zero_division=0)

print(f"precision: {precision_metric_threshold:.3f}; recall: {recall_metric_threshold:.3f}; f1: {f1_metric_threshold:.3f}")

precision: 0.015; recall: 0.621; f1: 0.029


In [69]:
model_with_threshold.set_threshold(0.13)
y_pred_threshold2 = model_with_threshold.predict(X_test)

precision_metric_threshold = precision_score(y_test, y_pred_threshold2, zero_division=0)
recall_metric_threshold = recall_score(y_test, y_pred_threshold2, zero_division=0)
f1_metric_threshold = f1_score(y_test, y_pred_threshold2, zero_division=0)

print(f"precision: {precision_metric_threshold:.3f}; recall: {recall_metric_threshold:.3f}; f1: {f1_metric_threshold:.3f}")

precision: 0.107; recall: 0.199; f1: 0.139


In [70]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    class_weight='balanced',
    random_state=42,
    max_iter=1000,
    solver='liblinear'
)

lr_model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'liblinear'
,max_iter,1000
,multi_class,'deprecated'


In [71]:
y_pred_lr = lr_model.predict(X_test)

precision_metric_lr = precision_score(y_test, y_pred_lr, zero_division=0)
recall_metric_lr = recall_score(y_test, y_pred_lr, zero_division=0)
f1_metric_lr = f1_score(y_test, y_pred_lr, zero_division=0)

print(f"precision: {precision_metric_lr:.3f}; recall: {recall_metric_lr:.3f}; f1: {f1_metric_lr:.3f}")

precision: 0.033; recall: 0.304; f1: 0.059
